# InternVL3 Simple Key-Value Extraction

**Purpose:** Simplified single-image extraction demo for InternVL3 vision-language model

This stripped-down notebook:
- Loads a single image
- Displays the image
- Runs InternVL3 extraction
- Shows the extracted key-value pairs

Perfect for quick testing and understanding the extraction process.

In [ ]:
# ============================================================================
# MINIMAL IMPORTS
# ============================================================================

import sys
import warnings
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown
from PIL import Image

# Add parent directory to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import essential modules
from common.config import (
    INTERNVL3_MODEL_PATH,
    EXTRACTION_FIELDS,
    FIELD_COUNT,
    DATA_DIR,
    OUTPUT_DIR
)
from models.internvl3_processor import InternVL3Processor

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✅ Simple InternVL3 Key-Value Extractor Ready")
print(f"📋 Will extract {FIELD_COUNT} fields from business documents")

In [ ]:
# ============================================================================
# INITIALIZE INTERNVL3 MODEL
# ============================================================================

print("🚀 Initializing InternVL3 Model...")
print("=" * 50)

# Initialize processor
processor = InternVL3Processor(model_path=INTERNVL3_MODEL_PATH)

print("\n✅ Model loaded successfully!")
print(f"📍 Model path: {INTERNVL3_MODEL_PATH}")
print(f"🎯 Ready to extract {FIELD_COUNT} business fields")

# Show extraction fields
print("\n📋 Fields to extract:")
for i, field in enumerate(EXTRACTION_FIELDS, 1):
    print(f"  {i:2d}. {field}")
    if i == 5:  # Show first 5 fields then ellipsis
        print(f"  ... and {FIELD_COUNT - 5} more fields")
        break

In [ ]:
# ============================================================================
# SELECT IMAGE TO PROCESS
# ============================================================================

# Default to first sample invoice
# CHANGE THIS PATH to test different images
image_path = Path(DATA_DIR) / "synthetic_invoice_001.jpeg"

# Alternative: specify your own image path
# image_path = Path("/path/to/your/image.jpeg")

# Verify image exists
if not image_path.exists():
    print(f"❌ Image not found: {image_path}")
    print(f"\n📁 Available images in {DATA_DIR}:")
    for img in sorted(Path(DATA_DIR).glob("*.jpeg"))[:5]:
        print(f"   - {img.name}")
else:
    print(f"✅ Selected image: {image_path.name}")
    print(f"📁 Full path: {image_path}")

In [ ]:
# ============================================================================
# DISPLAY THE IMAGE
# ============================================================================

if image_path.exists():
    # Load image with PIL to get dimensions
    pil_image = Image.open(image_path)
    width, height = pil_image.size
    
    print(f"📐 Image dimensions: {width} x {height} pixels")
    print(f"📄 Image type: {pil_image.format}")
    print(f"🎨 Image mode: {pil_image.mode}")
    
    # Display the image
    print("\n🖼️ Document Image:")
    display(IPImage(str(image_path), width=600))  # Adjust width as needed
else:
    print("❌ Cannot display - image file not found")

In [ ]:
# ============================================================================
# RUN EXTRACTION
# ============================================================================

print("🔍 Running InternVL3 Extraction...")
print("=" * 50)

# Process the image
result = processor.process_single_image(str(image_path))

# Extract key information
extracted_data = result['extracted_data']
processing_time = result['processing_time']
extracted_count = result['extracted_fields_count']
raw_response = result['raw_response']

print(f"\n✅ Extraction completed in {processing_time:.2f} seconds")
print(f"📊 Extracted {extracted_count}/{FIELD_COUNT} fields with values")
print(f"📝 Response length: {len(raw_response)} characters")

In [ ]:
# ============================================================================
# DISPLAY EXTRACTED KEY-VALUE PAIRS
# ============================================================================

display(Markdown("## 📋 Extracted Key-Value Pairs"))

# Group fields by extraction status
extracted_fields = {}
missing_fields = []

for field in EXTRACTION_FIELDS:
    value = extracted_data.get(field, "N/A")
    if value != "N/A" and value:
        extracted_fields[field] = value
    else:
        missing_fields.append(field)

# Display extracted fields
if extracted_fields:
    display(Markdown("### ✅ Successfully Extracted Fields:"))
    for field, value in extracted_fields.items():
        # Highlight important fields
        if field in ["TOTAL", "ABN", "INVOICE_NUMBER"]:
            print(f"⭐ {field:<25} : {value}")
        else:
            print(f"   {field:<25} : {value}")

# Display missing fields
if missing_fields:
    display(Markdown("\n### ❌ Fields Not Found:"))
    for field in missing_fields:
        print(f"   {field:<25} : N/A")

# Summary statistics
display(Markdown("\n### 📊 Extraction Summary:"))
print(f"Total fields expected:    {FIELD_COUNT}")
print(f"Fields with values:       {len(extracted_fields)}")
print(f"Missing fields:           {len(missing_fields)}")
print(f"Extraction rate:          {(len(extracted_fields)/FIELD_COUNT)*100:.1f}%")
print(f"Processing time:          {processing_time:.2f} seconds")

In [ ]:
# ============================================================================
# OPTIONAL: VIEW RAW RESPONSE
# ============================================================================

# Uncomment the lines below to see the raw model response

# display(Markdown("## 📝 Raw Model Response"))
# print("Raw output from InternVL3:")
# print("=" * 50)
# print(raw_response)
# print("=" * 50)

In [ ]:
# ============================================================================
# QUICK TEST WITH DIFFERENT IMAGE
# ============================================================================

# This cell allows you to quickly test another image without re-running everything

def quick_extract(image_file):
    """Quick extraction function for testing multiple images."""
    print(f"\n🔍 Processing: {Path(image_file).name}")
    print("=" * 50)
    
    # Check if file exists
    if not Path(image_file).exists():
        print(f"❌ File not found: {image_file}")
        return
    
    # Process image
    result = processor.process_single_image(str(image_file))
    
    # Extract key information
    extracted_data = result['extracted_data']
    processing_time = result['processing_time']
    extracted_count = result['extracted_fields_count']
    
    print(f"\n✅ Extraction completed in {processing_time:.2f} seconds")
    print(f"📊 Extracted {extracted_count}/{FIELD_COUNT} fields with values\n")
    
    # Group fields by extraction status
    extracted_fields = {}
    missing_fields = []
    
    for field in EXTRACTION_FIELDS:
        value = extracted_data.get(field, "N/A")
        if value != "N/A" and value:
            extracted_fields[field] = value
        else:
            missing_fields.append(field)
    
    # Display all extracted fields
    if extracted_fields:
        print("✅ Successfully Extracted Fields:")
        print("-" * 50)
        for field, value in extracted_fields.items():
            # Highlight important fields
            if field in ["TOTAL", "ABN", "INVOICE_NUMBER"]:
                print(f"⭐ {field:<25} : {value}")
            else:
                print(f"   {field:<25} : {value}")
    
    # Display missing fields
    if missing_fields:
        print("\n❌ Fields Not Found:")
        print("-" * 50)
        for field in missing_fields:
            print(f"   {field:<25} : N/A")
    
    # Summary statistics
    print("\n📊 Extraction Summary:")
    print("-" * 50)
    print(f"Total fields expected:    {FIELD_COUNT}")
    print(f"Fields with values:       {len(extracted_fields)}")
    print(f"Missing fields:           {len(missing_fields)}")
    print(f"Extraction rate:          {(len(extracted_fields)/FIELD_COUNT)*100:.1f}%")
    print(f"Processing time:          {processing_time:.2f} seconds")
    
    return extracted_data

# Example: Test with another invoice
# Uncomment and modify the path below to test
# test_result = quick_extract(Path(DATA_DIR) / "synthetic_invoice_002.jpeg")

In [ ]:
# ============================================================================
# BATCH PROCESS ALL JPEGS AND CREATE CSV (WITH VRAM MANAGEMENT)
# ============================================================================

import pandas as pd
from datetime import datetime
import torch
import gc

print("🚀 Batch Processing All JPEG Images with VRAM Management")
print("=" * 50)

# Import GPU optimization utilities
from common.gpu_optimization import (
    clear_model_caches,
    comprehensive_memory_cleanup,
    handle_memory_fragmentation
)

# Find all JPEG files in DATA_DIR
jpeg_files = sorted(Path(DATA_DIR).glob("*.jpeg"))
print(f"📁 Found {len(jpeg_files)} JPEG files to process")

# Determine batch size based on model
is_8b = "8B" in str(INTERNVL3_MODEL_PATH)
batch_size = 1 if is_8b else 2  # Conservative batch sizes for V100
print(f"🎯 Using batch size: {batch_size} (Model: {'8B' if is_8b else '2B'})")
print(f"⚡ VRAM optimization: ENABLED\n")

if jpeg_files:
    # Initialize results list
    all_results = []
    
    # Process images in batches for better memory management
    for batch_idx in range(0, len(jpeg_files), batch_size):
        batch_end = min(batch_idx + batch_size, len(jpeg_files))
        batch_files = jpeg_files[batch_idx:batch_end]
        
        print(f"\n[Batch {batch_idx//batch_size + 1}] Processing images {batch_idx + 1}-{batch_end} of {len(jpeg_files)}")
        
        # Pre-batch memory cleanup
        if batch_idx > 0:  # Skip cleanup before first batch
            handle_memory_fragmentation(threshold_gb=1.0, aggressive=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
        
        # Process each image in the batch
        for image_file in batch_files:
            print(f"  📄 Processing: {image_file.name}")
            
            try:
                # Process the image with proper error handling
                result = processor.process_single_image(str(image_file))
                extracted_data = result['extracted_data']
                
                # Create row dictionary with image name first
                row = {'image_name': image_file.name}
                
                # Add all fields in alphabetical order
                for field in sorted(EXTRACTION_FIELDS):
                    row[field] = extracted_data.get(field, "N/A")
                
                all_results.append(row)
                
                # Show quick stats
                extracted_count = sum(1 for v in extracted_data.values() if v != "N/A")
                print(f"     ✅ Extracted {extracted_count}/{FIELD_COUNT} fields")
                
            except torch.cuda.OutOfMemoryError as oom_error:
                print(f"     ⚠️ OOM Error - Attempting recovery...")
                
                # Emergency cleanup
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                clear_model_caches(processor.model, processor.tokenizer)
                handle_memory_fragmentation(threshold_gb=0.5, aggressive=True)
                gc.collect()
                
                # Add error row with all N/A values
                row = {'image_name': image_file.name}
                for field in sorted(EXTRACTION_FIELDS):
                    row[field] = "ERROR_OOM"
                all_results.append(row)
                
            except Exception as e:
                print(f"     ❌ Error: {e}")
                # Add error row with all N/A values
                row = {'image_name': image_file.name}
                for field in sorted(EXTRACTION_FIELDS):
                    row[field] = "N/A"
                all_results.append(row)
        
        # Post-batch cleanup - CRITICAL for V100
        if torch.cuda.is_available():
            # Comprehensive cleanup after each batch
            comprehensive_memory_cleanup(processor.model, processor.tokenizer)
            
            # Extra aggressive cleanup for 8B model
            if is_8b:
                handle_memory_fragmentation(threshold_gb=0.5, aggressive=True)
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
                print(f"  🧹 Aggressive memory cleanup after batch {batch_idx//batch_size + 1}")
            else:
                print(f"  🧹 Memory cleanup after batch {batch_idx//batch_size + 1}")
        
        # Force garbage collection
        gc.collect()
    
    # Create DataFrame
    print("\n📊 Creating DataFrame...")
    df = pd.DataFrame(all_results)
    
    # Ensure columns are in correct order: image_name first, then fields alphabetically
    column_order = ['image_name'] + sorted(EXTRACTION_FIELDS)
    df = df[column_order]
    
    # Generate timestamp for unique filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"internvl3_extraction_results_{timestamp}.csv"
    csv_path = Path(OUTPUT_DIR) / csv_filename
    
    # Ensure output directory exists
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    
    # Save to CSV
    df.to_csv(csv_path, index=False)
    print(f"\n💾 CSV saved to: {csv_path}")
    print(f"   Filename: {csv_filename}")
    
    # Display summary statistics
    print("\n📈 Extraction Summary:")
    print("-" * 50)
    print(f"Total images processed: {len(jpeg_files)}")
    print(f"Total columns in CSV: {len(df.columns)} (image_name + {FIELD_COUNT} fields)")
    
    # Count OOM errors if any
    oom_count = df.apply(lambda row: (row == "ERROR_OOM").any(), axis=1).sum()
    if oom_count > 0:
        print(f"⚠️ OOM errors encountered: {oom_count} images")
    
    # Calculate field extraction rates (excluding OOM errors)
    field_stats = {}
    for field in sorted(EXTRACTION_FIELDS):
        valid_rows = df[field].apply(lambda x: x not in ["N/A", "ERROR_OOM"])
        extracted = valid_rows.sum()
        total_valid = df[field].apply(lambda x: x != "ERROR_OOM").sum()
        if total_valid > 0:
            field_stats[field] = (extracted / total_valid) * 100
        else:
            field_stats[field] = 0
    
    # Show top and bottom performing fields
    sorted_fields = sorted(field_stats.items(), key=lambda x: x[1], reverse=True)
    
    print("\n🏆 Top 5 Best Extracted Fields:")
    for field, rate in sorted_fields[:5]:
        print(f"   {field:<25} : {rate:.1f}% success rate")
    
    print("\n⚠️ Top 5 Most Challenging Fields:")
    for field, rate in sorted_fields[-5:]:
        print(f"   {field:<25} : {rate:.1f}% success rate")
    
    # Display first few rows of DataFrame
    print("\n📋 Preview of Results (first 3 rows):")
    print(df.head(3).to_string())
    
    # Final memory cleanup
    if torch.cuda.is_available():
        print("\n🧹 Final memory cleanup...")
        comprehensive_memory_cleanup(processor.model, processor.tokenizer)
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
        # Show final memory status
        free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
        print(f"✅ Free VRAM after processing: {free_memory / 1e9:.2f} GB")
    
else:
    print("❌ No JPEG files found in DATA_DIR")
    print(f"   Please check: {DATA_DIR}")